In [ ]:
%pip install jedi

In [ ]:
import pandas
import numpy
import google.auth

print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("Google Auth:", google.auth.__version__)

%pip check


In [ ]:
%pip install -q "google-genai>=2.12.1,<3"

In [ ]:
import google.genai
import importlib.metadata

print("Gemini SDK version:", importlib.metadata.version("google-genai"))
print("Gemini SDK imported successfully!")

In [ ]:
for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Say hello! I am building an Autonomous AI Data Analyst."
)

print(response.text)

In [ ]:
import pandas as pd
import glob

csv_files = glob.glob("/content/*.csv")

print("CSV files found:", csv_files)

if csv_files:
    df = pd.read_csv(csv_files[0])
    print("\nDataset loaded successfully!")
    print("Dataset shape:", df.shape)
    display(df.head())
else:
    print("No CSV file found. Check the Colab Files panel.")

In [ ]:

# Give Gemini information about the uploaded dataset

dataset_info = f"""
Dataset columns and data types:
{df.dtypes.to_string()}

Dataset size:
{df.shape}

First 5 rows:
{df.head().to_string(index=False)}
"""

prompt = f"""
You are an autonomous AI Data Analyst.

Examine this dataset information:

{dataset_info}

Perform the following tasks independently:

1. Identify numerical and categorical columns.
2. Identify potential date/time columns.
3. Explain what this dataset represents.
4. Recommend important data quality checks.
5. Decide which business analyses should be performed.
6. Recommend appropriate visualizations.

Explain your findings clearly.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print(response.text)


In [ ]:

import json

available_tools = {
    "data_quality": "Check missing values and duplicates",
    "summary_statistics": "Calculate numerical statistics",
    "sales_by_category": "Analyze sales by product category",
    "sales_by_branch": "Compare sales across branches",
    "sales_trend": "Analyze sales over time"
}

prompt = f"""
You are an autonomous AI data analyst.

Dataset columns:
{df.dtypes.to_string()}

Available analytical tools:
{json.dumps(available_tools, indent=2)}

Choose the ONE most important analysis to execute first.

Return only a JSON object in this format:
{{
    "selected_tool": "tool_name",
    "reason": "Explain why you selected it"
}}
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print(response.text)


In [ ]:

import json

def data_quality(df):
    return {
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "missing_values": df.isnull().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum())
    }

# Extract Gemini's decision
decision = json.loads(
    response.text.strip().removeprefix("```json").removesuffix("```").strip()
)

# Register available executable tools
tool_registry = {
    "data_quality": data_quality
}

# Execute the AI-selected tool
selected_tool = decision["selected_tool"]

if selected_tool in tool_registry:
    result = tool_registry[selected_tool](df)

    print("AI SELECTED:", selected_tool)
    print("REASON:", decision["reason"])
    print("\nANALYSIS RESULTS:")
    print(json.dumps(result, indent=2))
else:
    print("Tool not registered yet:", selected_tool)


In [ ]:

interpretation_prompt = f"""
You are an autonomous AI Data Analyst.

A Python data quality tool has executed and returned
these verified results:

{json.dumps(result, indent=2)}

Your tasks:

1. Explain the data quality findings.
2. Identify any remaining data quality checks.
3. State whether the dataset is ready for basic analysis.
4. Recommend the next analytical operation.

Important:
- Base your conclusions only on the supplied results.
- Do not invent statistics.
- Do not claim the dataset is completely clean.
- Keep your explanation concise.
"""

interpretation = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=interpretation_prompt
)

print(interpretation.text)


In [ ]:
%pip install -q langgraph

In [ ]:

from typing import TypedDict, Any
from langgraph.graph import StateGraph, END

class AnalystState(TypedDict):
    dataset_info: str
    selected_tool: str
    reason: str
    analysis_results: dict[str, Any]
    completed_tools: list[str]
    final_report: str

print("LangGraph imported successfully!")
print("Analyst state created successfully!")


In [ ]:

from google.genai import types
import json

def analysis_planner(state: AnalystState):
    tools = {
        "data_quality": "Check missing values and duplicates",
        "summary_statistics": "Calculate numerical statistics",
        "sales_by_category": "Analyze sales by product category",
        "sales_by_branch": "Compare sales across branches",
        "sales_trend": "Analyze sales over time"
    }

    remaining_tools = {
        name: description
        for name, description in tools.items()
        if name not in state["completed_tools"]
    }

    if not remaining_tools:
        return {
            "selected_tool": "finish",
            "reason": "All available analyses completed."
        }

    prompt = f"""
    You are an autonomous AI data analyst.

    Dataset information:
    {state["dataset_info"]}

    Completed analyses:
    {state["completed_tools"]}

    Previous results:
    {json.dumps(state["analysis_results"], default=str)}

    Available tools:
    {json.dumps(remaining_tools)}

    Select the most useful analysis to perform next.

    Return a JSON object containing:
    selected_tool and reason.

    Choose only from the available tools.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    decision = json.loads(response.text)

    if decision.get("selected_tool") not in remaining_tools:
        raise ValueError("Gemini selected an unavailable tool.")

    print("AI selected:", decision["selected_tool"])
    print("Reason:", decision["reason"])

    return {
        "selected_tool": decision["selected_tool"],
        "reason": decision["reason"]
    }

print("AI Analysis Planner created successfully!")


In [ ]:

def execute_analysis(state: AnalystState):
    selected_tool = state["selected_tool"]

    def summary_statistics(df):
        return df.describe().round(2).to_dict()

    def sales_by_category(df):
        return (
            df.groupby("Product line")["Sales"]
            .sum()
            .round(2)
            .to_dict()
        )

    def sales_by_branch(df):
        return (
            df.groupby("Branch")["Sales"]
            .sum()
            .round(2)
            .to_dict()
        )

    def sales_trend(df):
        dates = pd.to_datetime(df["Date"], errors="coerce")

        return (
            df.assign(parsed_date=dates)
            .dropna(subset=["parsed_date"])
            .groupby("parsed_date")["Sales"]
            .sum()
            .round(2)
            .rename_axis("Date")
            .to_dict()
        )

    tool_registry = {
        "data_quality": data_quality,
        "summary_statistics": summary_statistics,
        "sales_by_category": sales_by_category,
        "sales_by_branch": sales_by_branch,
        "sales_trend": sales_trend
    }

    if selected_tool not in tool_registry:
        raise ValueError(f"Unknown analytical tool: {selected_tool}")

    result = tool_registry[selected_tool](df)

    updated_results = dict(state["analysis_results"])
    updated_results[selected_tool] = result

    completed = state["completed_tools"] + [selected_tool]

    print(f"Executed: {selected_tool}")

    return {
        "analysis_results": updated_results,
        "completed_tools": completed
    }

print("Analytical Tool Executor created successfully!")


In [ ]:

from langgraph.graph import StateGraph, START, END

# Create the workflow
workflow = StateGraph(AnalystState)

# Register the AI planner and tool executor
workflow.add_node("planner", analysis_planner)
workflow.add_node("executor", execute_analysis)

# Start with AI planning
workflow.add_edge(START, "planner")

# Decide whether to execute or finish
def route_decision(state: AnalystState):
    if state["selected_tool"] == "finish":
        return END
    return "executor"

workflow.add_conditional_edges(
    "planner",
    route_decision
)

# After execution, return to AI planning
workflow.add_edge("executor", "planner")

# Compile the autonomous agent
analyst_agent = workflow.compile()

print("Autonomous LangGraph workflow created successfully!")


In [ ]:

# Prepare dataset information for Gemini
dataset_info = f"""
Dataset shape: {df.shape}

Columns and data types:
{df.dtypes.to_string()}

Sample records:
{df.head(3).to_string(index=False)}
"""

# Initialize the agent's state
initial_state = {
    "dataset_info": dataset_info,
    "selected_tool": "",
    "reason": "",
    "analysis_results": {},
    "completed_tools": [],
    "final_report": ""
}

# Execute the autonomous workflow
final_state = analyst_agent.invoke(
    initial_state,
    config={"recursion_limit": 15}
)

print("\n=== AUTONOMOUS ANALYSIS COMPLETED ===")
print("Completed tools:", final_state["completed_tools"])


In [ ]:

import json
import pandas as pd

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {
            str(key): make_json_safe(value)
            for key, value in obj.items()
        }
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe(item) for item in obj]
    else:
        return obj

# Convert analysis results into JSON-safe format
safe_results = make_json_safe(
    final_state["analysis_results"]
)

# Test JSON conversion
results_json = json.dumps(
    safe_results,
    indent=2,
    default=str
)

print("JSON conversion successful!")
print("Available analyses:", list(safe_results.keys()))


In [ ]:

from google.genai import types
import json

chart_prompt = f"""
You are an autonomous AI Data Analyst.

Dataset columns and types:
{df.dtypes.to_string()}

Completed analytical results:
{results_json}

Choose ONE meaningful visualization.

Available chart types:
- bar
- line
- histogram
- scatter
- box

Return a JSON object with:
- chart_type
- x_column
- y_column
- reason

Rules:
1. Use only existing dataset columns.
2. For bar and line charts, aggregate numerical values using sum.
3. For histograms, set y_column to null.
4. Prefer visualizations that reveal useful business insights.
5. Do not invent data.
"""

chart_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=chart_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
)

chart_decision = json.loads(chart_response.text)

print("AI SELECTED VISUALIZATION:")
print(json.dumps(chart_decision, indent=2))


In [ ]:

import plotly.express as px
import pandas as pd

chart_type = chart_decision["chart_type"]
x_col = chart_decision["x_column"]
y_col = chart_decision["y_column"]

# Validate AI-selected columns
if x_col not in df.columns:
    raise ValueError(f"Invalid column: {x_col}")

if y_col is not None and y_col not in df.columns:
    raise ValueError(f"Invalid column: {y_col}")

# Execute the AI-selected visualization
if chart_type == "bar":
    chart_data = df.groupby(
        x_col, as_index=False
    )[y_col].sum()

    fig = px.bar(
        chart_data,
        x=x_col,
        y=y_col,
        color=x_col,
        title=f"Total {y_col} by {x_col}"
    )

elif chart_type == "line":
    chart_data = df.copy()

    if x_col == "Date":
        chart_data[x_col] = pd.to_datetime(
            chart_data[x_col],
            errors="coerce"
        )

    chart_data = chart_data.groupby(
        x_col, as_index=False
    )[y_col].sum()

    fig = px.line(
        chart_data,
        x=x_col,
        y=y_col,
        title=f"{y_col} Trend by {x_col}"
    )

elif chart_type == "histogram":
    fig = px.histogram(
        df,
        x=x_col,
        title=f"Distribution of {x_col}"
    )

elif chart_type == "scatter":
    fig = px.scatter(
        df,
        x=x_col,
        y=y_col,
        title=f"{x_col} vs {y_col}"
    )

elif chart_type == "box":
    fig = px.box(
        df,
        x=x_col,
        y=y_col,
        title=f"{y_col} by {x_col}"
    )

else:
    raise ValueError("Unsupported chart type")

fig.show()

print("AI-selected visualization generated successfully!")


In [ ]:

def visualization_node(state: AnalystState):
    import json
    import plotly.express as px
    from google.genai import types

    prompt = f"""
    You are an autonomous AI Data Analyst.

    Dataset columns and types:
    {df.dtypes.to_string()}

    Completed analyses:
    {list(state["analysis_results"].keys())}

    Choose ONE useful visualization.

    Available chart types:
    bar, line, histogram, scatter, box

    Return JSON containing:
    chart_type, x_column, y_column, reason

    Use only existing dataset columns.
    For a histogram, y_column must be null.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    decision = json.loads(response.text)

    chart_type = decision["chart_type"]
    x = decision["x_column"]
    y = decision["y_column"]

    if x not in df.columns or (y is not None and y not in df.columns):
        raise ValueError("AI selected invalid columns.")

    if chart_type == "bar":
        chart_data = df.groupby(x, as_index=False)[y].sum()
        fig = px.bar(chart_data, x=x, y=y)

    elif chart_type == "line":
        chart_data = df.copy()
        if x == "Date":
            chart_data[x] = pd.to_datetime(
                chart_data[x], errors="coerce"
            )
        chart_data = chart_data.groupby(x, as_index=False)[y].sum()
        fig = px.line(chart_data, x=x, y=y)

    elif chart_type == "histogram":
        fig = px.histogram(df, x=x)

    elif chart_type == "scatter":
        fig = px.scatter(df, x=x, y=y)

    elif chart_type == "box":
        fig = px.box(df, x=x, y=y)

    else:
        raise ValueError("Unsupported chart type.")

    fig.update_layout(
        title=f"AI Generated: {decision['reason']}"
    )

    fig.show()

    print("AI visualization generated successfully!")

    return {
        "reason": decision["reason"]
    }

print("LangGraph Visualization Node created successfully!")


In [ ]:

from langgraph.graph import StateGraph, START, END

# Create an updated workflow
workflow = StateGraph(AnalystState)

# Register nodes
workflow.add_node("planner", analysis_planner)
workflow.add_node("executor", execute_analysis)
workflow.add_node("visualization", visualization_node)

# Start with Gemini planning
workflow.add_edge(START, "planner")

# Route to executor or visualization
def route_decision(state: AnalystState):
    if state["selected_tool"] == "finish":
        return "visualization"
    return "executor"

workflow.add_conditional_edges(
    "planner",
    route_decision
)

# Return to AI after each analysis
workflow.add_edge("executor", "planner")

# Finish after generating visualization
workflow.add_edge("visualization", END)

# Compile updated agent
analyst_agent = workflow.compile()

print("Visualization integrated into LangGraph successfully!")


In [ ]:

# Initialize the agent with your uploaded dataset

initial_state = {
    "dataset_info": f"""
    Dataset shape: {df.shape}
    Column types:
    {df.dtypes.to_string()}
    """,
    "selected_tool": "",
    "reason": "",
    "analysis_results": {},
    "completed_tools": [],
    "final_report": ""
}

# Run the updated autonomous agent
final_state = analyst_agent.invoke(
    initial_state,
    config={"recursion_limit": 20}
)

print("\n=== AUTONOMOUS WORKFLOW COMPLETED ===")
print("Analyses performed:", final_state["completed_tools"])
print("Visualization:", final_state["reason"])


In [ ]:

from IPython.display import Markdown, display

report_prompt = f"""
You are an expert business intelligence analyst.

Dataset information:
{final_state["dataset_info"]}

Verified Python analysis results:
{results_json}

Generate a professional business intelligence report.

Include these sections:
1. Executive Summary
2. Dataset Overview
3. Data Quality Assessment
4. Key Analytical Findings
5. Business Recommendations
6. Limitations and Further Analysis

Requirements:
- Use only the provided calculated results.
- Include numerical evidence for important findings.
- Do not invent statistics or unsupported trends.
- Distinguish revenue from profit.
- Explain findings in clear business language.
- Use Markdown formatting.
"""

report_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=report_prompt
)

final_report = report_response.text

display(Markdown(final_report))


In [ ]:

from IPython.display import Markdown, display
from google.genai import types
import json

def report_generation_node(state: AnalystState):

    # Convert analytical results into JSON-safe format
    safe_data = make_json_safe(
        state["analysis_results"]
    )

    report_prompt = f"""
    You are an expert AI Business Intelligence Analyst.

    Dataset information:
    {state["dataset_info"]}

    Verified Python analytical results:
    {json.dumps(safe_data, default=str)}

    Generate a professional analytical report with:

    1. Executive Summary
    2. Dataset Overview
    3. Data Quality Assessment
    4. Key Statistical Findings
    5. Business Insights
    6. Recommendations
    7. Limitations

    Rules:
    - Base numerical claims only on supplied results.
    - Do not invent statistics.
    - Do not confuse revenue with profit.
    - Clearly identify limitations.
    - Use Markdown formatting.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=report_prompt
    )

    report = response.text

    print("AI Business Intelligence Report Generated!")

    display(Markdown(report))

    return {
        "final_report": report
    }

print("LangGraph Report Generator Node created successfully!")


In [ ]:

from langgraph.graph import StateGraph, START, END

# Create the complete workflow
workflow = StateGraph(AnalystState)

# Register all four nodes
workflow.add_node("planner", analysis_planner)
workflow.add_node("executor", execute_analysis)
workflow.add_node("visualization", visualization_node)
workflow.add_node("report", report_generation_node)

# Start with AI planning
workflow.add_edge(START, "planner")

# Route based on Gemini's decision
def route_decision(state: AnalystState):
    if state["selected_tool"] == "finish":
        return "visualization"
    return "executor"

workflow.add_conditional_edges(
    "planner",
    route_decision
)

# Continue analysis until completed
workflow.add_edge("executor", "planner")

# Generate report after visualization
workflow.add_edge("visualization", "report")

# Finish after report generation
workflow.add_edge("report", END)

# Compile complete autonomous agent
analyst_agent = workflow.compile()

print("Complete Autonomous AI Data Analyst workflow created!")


In [ ]:

# Prepare the initial state
initial_state = {
    "dataset_info": f"""
Dataset shape: {df.shape}

Column names and data types:
{df.dtypes.to_string()}
""",
    "selected_tool": "",
    "reason": "",
    "analysis_results": {},
    "completed_tools": [],
    "final_report": ""
}

# Execute the complete AI agent
final_state = analyst_agent.invoke(
    initial_state,
    config={"recursion_limit": 20}
)

# Display execution summary
print("\n=== AUTONOMOUS AI ANALYSIS COMPLETED ===")
print("Tools executed:", final_state["completed_tools"])
print("Report generated:", bool(final_state["final_report"]))
print("Report length:", len(final_state["final_report"]))


In [ ]:

import pandas as pd

def universal_dataset_profiler(df):
    profile = {
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "numerical_columns": df.select_dtypes(
            include="number"
        ).columns.tolist(),
        "categorical_columns": df.select_dtypes(
            include=["object", "category", "string"]
        ).columns.tolist(),
        "datetime_columns": df.select_dtypes(
            include=["datetime", "datetimetz"]
        ).columns.tolist(),
        "missing_values": df.isnull().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "column_types": {
            str(col): str(dtype)
            for col, dtype in df.dtypes.items()
        }
    }

    return profile

# Test on the currently uploaded dataset
dataset_profile = universal_dataset_profiler(df)

print("Universal Dataset Profiler created successfully!")
print(json.dumps(dataset_profile, indent=2))


In [ ]:

import pandas as pd

def detect_date_columns(df):
    detected_dates = []

    for column in df.columns:
        series = df[column]

        # Already stored as a datetime type
        if pd.api.types.is_datetime64_any_dtype(series):
            detected_dates.append(column)
            continue

        # Only examine text-like columns
        if not (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
        ):
            continue

        sample = series.dropna().astype(str).head(100)

        if sample.empty:
            continue

        # Avoid treating ordinary numbers as dates
        if not sample.str.contains(
            r"[-/:]|[A-Za-z]{3,}",
            regex=True
        ).any():
            continue

        try:
            parsed = pd.to_datetime(
                sample,
                errors="coerce",
                format="mixed"
            )

            if parsed.notna().mean() >= 0.9:
                detected_dates.append(column)

        except (ValueError, TypeError):
            continue

    return detected_dates

date_columns = detect_date_columns(df)

print("Automatically detected date columns:")
print(date_columns)


In [ ]:

import json
from google.genai import types

# Build a dataset-independent profile
dataset_profile = universal_dataset_profiler(df)
dataset_profile["detected_date_columns"] = detect_date_columns(df)

# Ask Gemini to understand the dataset
prompt = f"""
You are an autonomous AI data analyst.

Analyze this dataset profile:

{json.dumps(dataset_profile, indent=2, default=str)}

Your tasks:
1. Identify the likely domain of the dataset.
2. Explain what the dataset represents.
3. Identify useful numerical and categorical variables.
4. Identify date/time columns, if available.
5. Recommend relevant analytical operations.
6. Identify potential data quality concerns.

Rules:
- Do not assume this is a sales dataset.
- Do not invent columns or statistics.
- Recommend analyses only for available columns.
- Return your findings as a JSON object.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
)

ai_dataset_understanding = json.loads(response.text)

print("=== AI DATASET UNDERSTANDING ===")
print(json.dumps(ai_dataset_understanding, indent=2))


In [ ]:

import pandas as pd

def universal_statistics(df):
    numerical_df = df.select_dtypes(include="number")

    if numerical_df.empty:
        return {
            "status": "skipped",
            "reason": "No numerical columns available."
        }

    statistics = numerical_df.describe().round(2)

    return {
        "status": "success",
        "columns_analyzed": numerical_df.columns.tolist(),
        "statistics": statistics.to_dict(),
        "correlations": (
            numerical_df.corr()
            .round(3)
            .to_dict()
            if numerical_df.shape[1] >= 2
            else {}
        )
    }

# Test the universal tool
stats_result = universal_statistics(df)

print("Universal statistical analysis completed!")
print("Columns analyzed:", stats_result.get("columns_analyzed"))
print("Status:", stats_result["status"])


In [ ]:

import pandas as pd

def universal_categorical_analysis(df):
    categorical_columns = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    results = {}

    for column in categorical_columns:
        series = df[column]
        unique_count = series.nunique(dropna=True)

        # Skip likely IDs and columns with too many unique values
        if unique_count > 50 or unique_count == len(df):
            continue

        counts = (
            series.value_counts(dropna=False)
            .head(10)
        )

        results[column] = {
            "unique_categories": int(unique_count),
            "top_categories": {
                str(key): int(value)
                for key, value in counts.items()
            }
        }

    if not results:
        return {
            "status": "skipped",
            "reason": "No suitable categorical columns found."
        }

    return {
        "status": "success",
        "categorical_analysis": results
    }

# Test the tool
categorical_result = universal_categorical_analysis(df)

print("Universal categorical analysis completed!")
print("Status:", categorical_result["status"])
print("Columns analyzed:",
      list(categorical_result.get("categorical_analysis", {}).keys()))


In [ ]:

import pandas as pd

def universal_grouped_analysis(df, category_col, numeric_col, operation="mean"):

    # Validate selected columns
    if category_col not in df.columns:
        return {"status": "error", "reason": "Invalid category column"}

    if numeric_col not in df.columns:
        return {"status": "error", "reason": "Invalid numeric column"}

    if not pd.api.types.is_numeric_dtype(df[numeric_col]):
        return {"status": "error", "reason": "Selected measure is not numerical"}

    if operation not in ["sum", "mean", "median", "count"]:
        return {"status": "error", "reason": "Unsupported operation"}

    # Perform grouped analysis
    grouped = (
        df.groupby(category_col, dropna=False)[numeric_col]
        .agg(operation)
        .sort_values(ascending=False)
        .head(20)
        .round(2)
    )

    return {
        "status": "success",
        "category_column": category_col,
        "numeric_column": numeric_col,
        "operation": operation,
        "results": {
            str(key): float(value)
            for key, value in grouped.items()
        }
    }

print("Universal Grouped Analysis Tool created successfully!")


In [ ]:

import json
from google.genai import types

# Identify suitable columns dynamically
categorical_cols = df.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_cols = df.select_dtypes(
    include="number"
).columns.tolist()

prompt = f"""
You are an autonomous AI Data Analyst.

Dataset profile:
{json.dumps(dataset_profile, default=str)}

Available categorical columns:
{categorical_cols}

Available numerical columns:
{numerical_cols}

Select ONE meaningful grouped analysis.

Return a JSON object with these fields:
- category_column
- numeric_column
- operation
- reason

Allowed operations: sum, mean, median, count.

Rules:
- Use only columns from the provided lists.
- Avoid identifiers and date/time columns.
- Prefer meaningful business or statistical insights.
- Do not assume the dataset is about sales.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
)

group_decision = json.loads(response.text)

print("=== GEMINI ANALYSIS DECISION ===")
print(json.dumps(group_decision, indent=2))


In [ ]:

# Validate Gemini's decision
category = group_decision["category_column"]
numeric = group_decision["numeric_column"]
operation = group_decision["operation"]

if category not in categorical_cols:
    raise ValueError("AI selected an invalid categorical column")

if numeric not in numerical_cols:
    raise ValueError("AI selected an invalid numerical column")

if operation not in ["sum", "mean", "median", "count"]:
    raise ValueError("AI selected an unsupported operation")

# Execute the selected analysis
group_result = universal_grouped_analysis(
    df,
    category_col=category,
    numeric_col=numeric,
    operation=operation
)

print("=== AI-SELECTED GROUPED ANALYSIS ===")
print("Category:", category)
print("Numerical measure:", numeric)
print("Operation:", operation)
print("AI reasoning:", group_decision["reason"])

print("\n=== ACTUAL ANALYSIS RESULTS ===")
print(json.dumps(group_result, indent=2))


In [ ]:

import pandas as pd

def universal_time_series(df, date_col, numeric_col, frequency="MS"):
    # Validate inputs
    if date_col not in df.columns or numeric_col not in df.columns:
        return {"status": "error", "reason": "Column not found"}

    if not pd.api.types.is_numeric_dtype(df[numeric_col]):
        return {"status": "error", "reason": "Measure must be numerical"}

    if frequency not in ["D", "W", "MS", "YS"]:
        return {"status": "error", "reason": "Unsupported frequency"}

    # Convert date values
    dates = pd.to_datetime(
        df[date_col],
        errors="coerce",
        format="mixed"
    )

    temp = pd.DataFrame({
        "date": dates,
        "value": df[numeric_col]
    }).dropna()

    if temp.empty:
        return {"status": "skipped", "reason": "No valid dates and values"}

    # Aggregate values by time period
    trend = (
        temp.set_index("date")["value"]
        .resample(frequency)
        .sum()
        .round(2)
    )

    return {
        "status": "success",
        "date_column": date_col,
        "numeric_column": numeric_col,
        "frequency": frequency,
        "results": {
            str(date.date()): float(value)
            for date, value in trend.items()
        }
    }

print("Universal Time-Series Analysis Tool created successfully!")


In [ ]:

# Register all dataset-independent analytical tools

universal_tools = {
    "data_quality": data_quality,
    "summary_statistics": universal_statistics,
    "categorical_analysis": universal_categorical_analysis,
    "grouped_analysis": universal_grouped_analysis,
    "time_series": universal_time_series
}

# Describe the inputs each tool requires
tool_parameters = {
    "data_quality": [],
    "summary_statistics": [],
    "categorical_analysis": [],
    "grouped_analysis": [
        "category_col",
        "numeric_col",
        "operation"
    ],
    "time_series": [
        "date_col",
        "numeric_col",
        "frequency"
    ]
}

print("Universal Tool Registry created successfully!")
print("Registered tools:", list(universal_tools.keys()))


In [ ]:

import json
from google.genai import types

def universal_analysis_planner(state: AnalystState):

    profile = universal_dataset_profiler(df)
    profile["detected_date_columns"] = detect_date_columns(df)

    remaining_tools = {
        name: {
            "description": {
                "data_quality": "Check missing values and duplicate rows",
                "summary_statistics": "Calculate numerical statistics and correlations",
                "categorical_analysis": "Analyze frequencies of categorical columns",
                "grouped_analysis": "Compare a numerical measure across categories",
                "time_series": "Analyze a numerical measure over time"
            }[name],
            "parameters": tool_parameters[name]
        }
        for name in universal_tools
        if name not in state["completed_tools"]
    }

    if not remaining_tools:
        return {
            "selected_tool": "finish",
            "reason": "All available tools have been considered."
        }

    prompt = f"""
    You are an autonomous data analyst.

    Dataset profile:
    {json.dumps(profile, default=str)}

    Completed analyses:
    {state["completed_tools"]}

    Available tools and parameters:
    {json.dumps(remaining_tools)}

    Choose ONE appropriate next analytical tool.

    Return JSON with:
    - selected_tool
    - parameters (JSON object)
    - reason

    Rules:
    - Select only from available tools.
    - Use only actual dataset column names.
    - Grouped analysis requires a categorical column and numeric column.
    - Time-series requires a detected date column and numeric column.
    - Choose operation from sum, mean, median, count.
    - Choose frequency from D, W, MS, YS.
    - Use an empty parameters object for tools that need no additional inputs.
    - Do not assume the dataset is about sales.
    - Prefer data_quality early.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    decision = json.loads(response.text)

    selected = decision.get("selected_tool")

    if selected not in remaining_tools:
        raise ValueError(f"Invalid tool selected: {selected}")

    parameters = decision.get("parameters", {})

    if not isinstance(parameters, dict):
        raise ValueError("Tool parameters must be a dictionary")

    # Store parameters with the decision for execution
    decision["parameters"] = parameters

    print("AI selected:", selected)
    print("Parameters:", parameters)
    print("Reason:", decision.get("reason", ""))

    return {
        "selected_tool": selected,
        "reason": decision.get("reason", ""),
        "tool_parameters": parameters
    }

print("Universal AI Analysis Planner created successfully!")


In [ ]:

from typing import TypedDict, Any

class AnalystState(TypedDict):
    dataset_info: str
    selected_tool: str
    tool_parameters: dict[str, Any]
    reason: str
    analysis_results: dict[str, Any]
    completed_tools: list[str]
    final_report: str

print("Universal Analyst State updated successfully!")


In [ ]:

def universal_tool_executor(state: AnalystState):
    selected = state["selected_tool"]
    params = state.get("tool_parameters", {})

    if selected not in universal_tools:
        raise ValueError(f"Unknown tool: {selected}")

    if not isinstance(params, dict):
        raise ValueError("Tool parameters must be a dictionary")

    required = tool_parameters[selected]

    # Verify required parameters
    missing = [p for p in required if p not in params]
    if missing:
        raise ValueError(f"Missing parameters: {missing}")

    # Reject unexpected parameters
    unexpected = [p for p in params if p not in required]
    if unexpected:
        raise ValueError(f"Unexpected parameters: {unexpected}")

    # Validate column selections
    for key in ("category_col", "numeric_col", "date_col"):
        if key in params and params[key] not in df.columns:
            raise ValueError(f"Invalid dataset column: {params[key]}")

    # Execute Gemini-selected analytical tool
    result = universal_tools[selected](df, **params)

    updated_results = dict(state["analysis_results"])
    updated_results[selected] = result

    completed = list(state["completed_tools"])
    completed.append(selected)

    print("Executed:", selected)
    print("Status:", result.get("status", "completed"))

    return {
        "analysis_results": updated_results,
        "completed_tools": completed
    }

print("Universal Tool Executor created successfully!")


In [ ]:

from langgraph.graph import StateGraph, START, END

# Create universal agent workflow
universal_workflow = StateGraph(AnalystState)

# Register nodes
universal_workflow.add_node(
    "planner", universal_analysis_planner
)
universal_workflow.add_node(
    "executor", universal_tool_executor
)
universal_workflow.add_node(
    "visualization", visualization_node
)
universal_workflow.add_node(
    "report", report_generation_node
)

# Start with Gemini planning
universal_workflow.add_edge(START, "planner")

# Route based on agent decision
def universal_route(state: AnalystState):
    if state["selected_tool"] == "finish":
        return "visualization"
    return "executor"

universal_workflow.add_conditional_edges(
    "planner", universal_route
)

# Continue planning after each analysis
universal_workflow.add_edge("executor", "planner")

# Generate visualization and report
universal_workflow.add_edge("visualization", "report")
universal_workflow.add_edge("report", END)

# Compile the universal agent
universal_agent = universal_workflow.compile()

print("Universal Autonomous AI Data Analyst created successfully!")


In [ ]:

from google.colab import files
import pandas as pd
import io

# Upload a new dataset
uploaded = files.upload()

# Read the uploaded CSV into df
filename = next(iter(uploaded))

df = pd.read_csv(
    io.BytesIO(uploaded[filename])
)

print("New dataset loaded:", filename)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


In [ ]:

# Prepare the new dataset's initial state
initial_state = {
    "dataset_info": f"""
Dataset shape: {df.shape}
Column types:
{df.dtypes.to_string()}
""",
    "selected_tool": "",
    "tool_parameters": {},
    "reason": "",
    "analysis_results": {},
    "completed_tools": [],
    "final_report": ""
}

# Run the universal agent
final_state = universal_agent.invoke(
    initial_state,
    config={"recursion_limit": 20}
)

print("\n=== UNIVERSAL AGENT TEST COMPLETED ===")
print("Tools executed:", final_state["completed_tools"])
print("Report generated:", bool(final_state["final_report"]))


In [ ]:

# Inspect the actual analytical evidence available to Gemini

print("=== VERIFIED ANALYSIS RESULTS ===")

for tool_name, tool_result in final_state["analysis_results"].items():
    print(f"\nTool: {tool_name}")

    if isinstance(tool_result, dict):
        print("Available result fields:", list(tool_result.keys()))

        if tool_name == "grouped_analysis":
            print("Grouped analysis:", tool_result)

        elif tool_name == "data_quality":
            print("Data quality:", tool_result)

    else:
        print("Result type:", type(tool_result).__name__)

print("\n=== REPORT VALIDATION CHECK ===")
print("Dataset rows:", len(df))
print("Report generated:", bool(final_state["final_report"]))


In [ ]:

import json
from IPython.display import Markdown, display

def verified_report_node(state: AnalystState):

    safe_results = make_json_safe(
        state["analysis_results"]
    )

    evidence = json.dumps(
        safe_results,
        indent=2,
        default=str
    )

    prompt = f"""
    You are a rigorous Business Intelligence Analyst.

    DATASET INFORMATION:
    {state["dataset_info"]}

    VERIFIED PYTHON RESULTS:
    {evidence}

    Produce a professional analytical report.

    STRICT RULES:
    1. Every numerical claim must be directly supported
       by the verified Python results.
    2. Do not invent correlation coefficients, trends,
       percentages, profits, or other statistics.
    3. Do not claim that data quality checks were performed
       unless their results are provided.
    4. Distinguish observations from assumptions.
    5. Do not infer causation from correlation.
    6. If evidence is insufficient, explicitly say so.
    7. Avoid unsupported business recommendations.
    8. Use valid Markdown formatting.

    REPORT SECTIONS:
    1. Executive Summary
    2. Dataset Overview
    3. Data Quality Assessment
    4. Verified Statistical Findings
    5. Business Insights
    6. Recommendations
    7. Limitations

    For important numerical findings, identify which
    Python analysis tool supports the conclusion.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    report = response.text

    display(Markdown(report))

    return {"final_report": report}

print("Evidence-Based Report Generator created successfully!")


In [ ]:

from langgraph.graph import StateGraph, START, END

# Rebuild workflow with evidence-based reporting
universal_workflow = StateGraph(AnalystState)

universal_workflow.add_node(
    "planner", universal_analysis_planner
)
universal_workflow.add_node(
    "executor", universal_tool_executor
)
universal_workflow.add_node(
    "visualization", visualization_node
)
universal_workflow.add_node(
    "report", verified_report_node
)

universal_workflow.add_edge(START, "planner")

def universal_route(state: AnalystState):
    if state["selected_tool"] == "finish":
        return "visualization"
    return "executor"

universal_workflow.add_conditional_edges(
    "planner", universal_route
)

universal_workflow.add_edge("executor", "planner")
universal_workflow.add_edge("visualization", "report")
universal_workflow.add_edge("report", END)

universal_agent = universal_workflow.compile()

print("Evidence-based reporting integrated successfully!")
print("Universal autonomous workflow updated!")


In [ ]:

from typing import TypedDict, Any

class QuestionAnswerState(TypedDict):
    user_question: str
    dataset_info: str
    selected_tool: str
    tool_parameters: dict[str, Any]
    analysis_result: dict[str, Any]
    final_answer: str

print("Ask Your Data agent state created successfully!")


In [ ]:

import json
from google.genai import types

def question_analysis_planner(state: QuestionAnswerState):

    profile = universal_dataset_profiler(df)
    profile["detected_date_columns"] = detect_date_columns(df)

    prompt = f"""
    You are an AI Data Analyst answering a user's question.

    USER QUESTION:
    {state["user_question"]}

    DATASET PROFILE:
    {json.dumps(profile, default=str)}

    AVAILABLE ANALYTICAL TOOLS:
    {json.dumps(tool_parameters)}

    Choose the best tool to answer the user's question.

    Return a JSON object with:
    - selected_tool
    - parameters
    - reason

    Rules:
    1. Use only tools from the available list.
    2. Use only actual dataset column names.
    3. For grouped_analysis, provide category_col,
       numeric_col, and operation.
    4. For time_series, provide date_col,
       numeric_col, and frequency.
    5. For tools without parameters, return {{}}.
    6. Do not invent column names or statistics.
    7. If the question cannot be answered using
       these tools, explain why.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    decision = json.loads(response.text)

    print("User question:", state["user_question"])
    print("AI selected tool:", decision.get("selected_tool"))
    print("Parameters:", decision.get("parameters"))

    return {
        "selected_tool": decision["selected_tool"],
        "tool_parameters": decision.get("parameters", {})
    }

print("Ask Your Data AI Planner created successfully!")


In [ ]:

def question_tool_executor(state: QuestionAnswerState):
    selected = state["selected_tool"]
    params = state.get("tool_parameters", {})

    # Ensure Gemini selected a registered tool
    if selected not in universal_tools:
        raise ValueError(f"Unsupported tool: {selected}")

    if not isinstance(params, dict):
        raise ValueError("Tool parameters must be a dictionary")

    # Validate required parameters
    required = tool_parameters[selected]

    missing = [p for p in required if p not in params]
    if missing:
        raise ValueError(f"Missing parameters: {missing}")

    # Reject unexpected parameters
    unexpected = [p for p in params if p not in required]
    if unexpected:
        raise ValueError(f"Unexpected parameters: {unexpected}")

    # Check that selected columns exist
    for key in ["category_col", "numeric_col", "date_col"]:
        if key in params and params[key] not in df.columns:
            raise ValueError(f"Invalid dataset column: {params[key]}")

    # Execute the selected Python tool
    result = universal_tools[selected](df, **params)

    print("Executed analytical tool:", selected)
    print("Execution status:", result.get("status", "completed"))

    return {
        "analysis_result": make_json_safe(result)
    }

print("Ask Your Data Tool Executor created successfully!")


In [ ]:

from IPython.display import Markdown, display
import json

def question_answer_generator(state: QuestionAnswerState):

    prompt = f"""
    You are an expert AI Data Analyst.

    USER QUESTION:
    {state["user_question"]}

    DATASET INFORMATION:
    {state["dataset_info"]}

    EXECUTED ANALYTICAL TOOL:
    {state["selected_tool"]}

    VERIFIED PYTHON RESULTS:
    {json.dumps(state["analysis_result"], indent=2, default=str)}

    Answer the user's question using these results.

    STRICT RULES:
    1. Use only the provided analytical results.
    2. Do not invent numbers or findings.
    3. Directly answer the user's question.
    4. Explain important findings clearly.
    5. Do not claim causation from correlation.
    6. If the results are insufficient, say so.
    7. Use clean Markdown formatting.
    8. Keep the answer concise unless the question
       requires a detailed explanation.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    answer = response.text

    print("AI ANSWER:")
    display(Markdown(answer))

    return {"final_answer": answer}

print("Ask Your Data Answer Generator created successfully!")


In [ ]:

from langgraph.graph import StateGraph, START, END

# Create the question-answering workflow
qa_workflow = StateGraph(QuestionAnswerState)

# Register the three components
qa_workflow.add_node(
    "planner", question_analysis_planner
)

qa_workflow.add_node(
    "executor", question_tool_executor
)

qa_workflow.add_node(
    "answer", question_answer_generator
)

# Connect the workflow
qa_workflow.add_edge(START, "planner")
qa_workflow.add_edge("planner", "executor")
qa_workflow.add_edge("executor", "answer")
qa_workflow.add_edge("answer", END)

# Compile the question-answering agent
qa_agent = qa_workflow.compile()

print("Ask Your Data Agent created successfully!")


In [ ]:

# Enter a question about your uploaded dataset
user_question = "Which category has the highest average profit?"

# Prepare the question-answering state
qa_initial_state = {
    "user_question": user_question,
    "dataset_info": f"""
Dataset shape: {df.shape}
Columns and data types:
{df.dtypes.to_string()}
""",
    "selected_tool": "",
    "tool_parameters": {},
    "analysis_result": {},
    "final_answer": ""
}

# Run the Ask Your Data agent
qa_result = qa_agent.invoke(qa_initial_state)

print("\n=== QUESTION ANSWERING COMPLETED ===")
print("Selected tool:", qa_result["selected_tool"])
print("Parameters:", qa_result["tool_parameters"])
print("Answer generated:", bool(qa_result["final_answer"]))


In [ ]:

# Interactive question input
user_question = input("Ask a question about your dataset: ").strip()

if not user_question:
    print("Please enter a question.")
else:
    qa_initial_state = {
        "user_question": user_question,
        "dataset_info": f"""
Dataset shape: {df.shape}
Columns and data types:
{df.dtypes.to_string()}
""",
        "selected_tool": "",
        "tool_parameters": {},
        "analysis_result": {},
        "final_answer": ""
    }

    qa_result = qa_agent.invoke(qa_initial_state)

    print("\n=== ANALYSIS COMPLETED ===")
    print("Selected tool:", qa_result["selected_tool"])
    print("Parameters:", qa_result["tool_parameters"])


In [ ]:

print("=== AVAILABLE DATASET COLUMNS ===")

for column in df.columns:
    print(column)

print("\n=== POSSIBLE COST-RELATED COLUMNS ===")

keywords = ["cost", "cogs", "profit", "sales", "price"]

matches = [
    column for column in df.columns
    if any(word in column.lower() for word in keywords)
]

print(matches)


In [ ]:

# Create a derived cost measure
df["Estimated Cost"] = df["Sales"] - df["Profit"]

# Calculate total sales and estimated cost by product
product_analysis = (
    df.groupby("Product Name")[
        ["Sales", "Estimated Cost"]
    ]
    .sum()
)

highest_sales = product_analysis["Sales"].idxmax()
highest_cost = product_analysis["Estimated Cost"].idxmax()

print("=== PRODUCT ANALYSIS ===")

print("\nHighest Sales Product:")
print(highest_sales)
print("Total Sales:", round(product_analysis.loc[highest_sales, "Sales"], 2))

print("\nHighest Estimated Cost Product:")
print(highest_cost)
print("Total Estimated Cost:", round(product_analysis.loc[highest_cost, "Estimated Cost"], 2))


In [ ]:

import json
from google.genai import types

def multi_step_question_planner(question, df):

    profile = universal_dataset_profiler(df)
    profile["detected_date_columns"] = detect_date_columns(df)

    prompt = f"""
    You are an autonomous AI Data Analyst.

    USER QUESTION:
    {question}

    DATASET PROFILE:
    {json.dumps(profile, default=str)}

    AVAILABLE ANALYTICAL TOOLS:
    {json.dumps(tool_parameters)}

    Your task is to break the user's question into
    the analytical operations required to answer it.

    Return JSON with:
    - tasks: a list of analytical tasks
    - explanation: brief reasoning

    Each task must contain:
    - question_part
    - selected_tool
    - parameters
    - reason

    Rules:
    1. Create separate tasks when multiple analyses are needed.
    2. Select only registered analytical tools.
    3. Use actual dataset column names.
    4. Do not invent missing columns.
    5. If a requested measure must be derived, explain
       the required calculation and assumptions.
    6. Do not execute calculations yourself.
    7. Do not assume the dataset is about sales.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)

print("Multi-Step Question Planner created successfully!")


In [ ]:

import json

# Test a question requiring multiple analyses
test_question = (
    "Which product has the highest total sales "
    "and which product has the highest total cost?"
)

# Ask Gemini to create an analysis plan
multi_step_plan = multi_step_question_planner(
    test_question,
    df
)

print("=== AI MULTI-STEP ANALYSIS PLAN ===")
print(json.dumps(multi_step_plan, indent=2))


In [ ]:

def multi_step_executor(plan, df):
    results = []

    tasks = plan.get("tasks", [])

    if not isinstance(tasks, list) or not tasks:
        raise ValueError("No valid analytical tasks found.")

    if len(tasks) > 5:
        raise ValueError("Maximum 5 tasks per question.")

    for index, task in enumerate(tasks, start=1):
        tool_name = task.get("selected_tool")
        params = task.get("parameters", {})

        if tool_name not in universal_tools:
            results.append({
                "task": index,
                "status": "skipped",
                "reason": f"Unsupported tool: {tool_name}"
            })
            continue

        if not isinstance(params, dict):
            raise ValueError("Invalid tool parameters.")

        required = tool_parameters[tool_name]

        if set(params) != set(required):
            results.append({
                "task": index,
                "status": "skipped",
                "reason": "Invalid or missing tool parameters"
            })
            continue

        try:
            result = universal_tools[tool_name](
                df, **params
            )

            results.append({
                "task": index,
                "question_part": task.get("question_part"),
                "tool": tool_name,
                "parameters": params,
                "result": make_json_safe(result)
            })

            print(f"Task {index} completed: {tool_name}")

        except (ValueError, TypeError, KeyError) as error:
            results.append({
                "task": index,
                "status": "error",
                "reason": str(error)
            })

    return results

print("Multi-Step Tool Executor created successfully!")


In [ ]:

import json

# Execute Gemini's multi-step analysis plan
multi_step_results = multi_step_executor(
    multi_step_plan,
    df
)

# Display the results of each task
print("\n=== MULTI-STEP ANALYSIS RESULTS ===")

for task in multi_step_results:
    print("\nTask:", task["task"])
    print("Question:", task.get("question_part", "N/A"))
    print("Tool:", task.get("tool", "N/A"))
    print("Status:", task.get(
        "status",
        task.get("result", {}).get("status", "completed")
    ))

    if "result" in task:
        print("Results:")
        print(json.dumps(task["result"], indent=2, default=str)[:2000])
    else:
        print("Reason:", task.get("reason", "Unknown"))



In [ ]:

import json
from IPython.display import Markdown, display

def multi_step_answer_generator(question, results):

    safe_results = make_json_safe(results)

    prompt = f"""
    You are an expert AI Data Analyst.

    USER QUESTION:
    {question}

    VERIFIED PYTHON ANALYSIS RESULTS:
    {json.dumps(safe_results, indent=2, default=str)}

    Answer every part of the user's question.

    STRICT RULES:
    1. Use only the supplied Python results.
    2. Do not invent numbers or conclusions.
    3. Answer each part of the question separately.
    4. If an analysis failed or was skipped, explain why.
    5. Clearly distinguish Sales, Profit, and Cost.
    6. Label derived costs as estimates.
    7. Do not claim that an unverified calculation succeeded.
    8. Use clear Markdown formatting.
    9. Keep the answer concise and informative.
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    answer = response.text

    display(Markdown(answer))

    return answer

print("Multi-Step AI Answer Generator created successfully!")


In [ ]:

# Generate a combined answer from completed analyses

multi_step_final_answer = multi_step_answer_generator(
    test_question,
    multi_step_results
)

print("\n=== MULTI-STEP QUESTION ANSWERING COMPLETED ===")
print("Answer generated:", bool(multi_step_final_answer))
print("Tasks processed:", len(multi_step_results))


In [ ]:

def ask_your_data(question, dataset):
    question = question.strip()

    if not question:
        return "Please enter a question."

    # Step 1: Gemini creates an analysis plan
    plan = multi_step_question_planner(
        question, dataset
    )

    print("=== AI ANALYSIS PLAN ===")
    print(json.dumps(plan, indent=2))

    # Step 2: Python executes the selected tools
    results = multi_step_executor(
        plan, dataset
    )

    # Step 3: Gemini interprets the results
    answer = multi_step_answer_generator(
        question, results
    )

    return answer

print("Multi-Step Ask Your Data function created successfully!")


In [ ]:

# Ask a new question about the uploaded dataset

question = input("Ask your dataset: ").strip()

if question:
    answer = ask_your_data(question, df)
else:
    print("Please enter a question.")


In [ ]:

from IPython.display import Markdown, display
import json

def multi_step_answer_generator(question, results):

    safe_results = make_json_safe(results)

    prompt = f"""
You are a professional AI Data Analyst.

USER QUESTION:
{question}

VERIFIED PYTHON RESULTS:
{json.dumps(safe_results, indent=2, default=str)}

Answer the user's question accurately.

STRICT RULES:
1. Use only supplied Python results.
2. Answer every part of the question.
3. Do not invent statistics.
4. Do not add unrelated information.
5. If a calculation failed, clearly explain why.
6. State when a value is estimated or derived.
7. Use plain text with simple headings.
8. Do not use Markdown bold markers or tables.
9. Format numbers with commas and two decimals.
10. Do not assume a currency unless the dataset
    explicitly identifies one.
11. Clearly identify the highest or lowest values
    when requested.
12. Keep the answer concise.
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    answer = response.text

    print("\n=== AI DATA ANALYSIS ANSWER ===")
    print(answer)

    return answer

print("Improved Multi-Step Answer Generator ready!")
